In [1]:
import os
import itertools
import numpy as np
import networkx as nx
from tqdm.notebook import tqdm
from multiprocessing import Pool

from ripser_count import get_barcodes
from stats_count import count_top_stats, adj_ms_to_nx_lists

In [2]:
np.random.seed(125)
stats_cap = 500
stats_name = "s_w_e_v_c_b0b1"
thresholds_array = [0.025, 0.05, 0.1, 0.25, 0.5, 0.75]
thrs = len(thresholds_array)

In [3]:
pth_ats = "/home/jovyan/hallu_data/hallu_detection_ats/helm/"

In [4]:
attention_files = [x for x in os.listdir(pth_ats) if "attentions" in x]

# generate filtering features for full output texts, one colour

In [39]:
def process_file(fname, pth_ats, thresholds_array, stats_name, save_fldr):
    id = fname.split("_")[0]
    try:
        ats = np.load(pth_ats + f"{id}_attentions.npy")
        token_ids = np.load(pth_ats + f"{id}_token_ids.npy")
        hidden_states = np.load(pth_ats + f"{id}_hidden_states.npy")
        params = np.load(pth_ats + f"{id}_params.npy")

        at_cropped = ats[:, :, params[0]:params[3], params[0]:params[3]]
        res = count_top_stats(at_cropped[None,], thresholds_array, [at_cropped.shape[2] for i in range(at_cropped.shape[2])], stats_name, 500, verbose=False)
        res = res.squeeze()
        np.save(save_fldr + f"{id}_full.npy", res)
    except Exception as e:
        print(f"failed for id {id}, error: {e}")

In [40]:
save_fldr = "/home/jovyan/hallu_data/hallu_detection_features/helm/filtrations/"

pool = Pool(10)
pool.starmap(process_file, [(fname, pth_ats, thresholds_array, stats_name, save_fldr) for fname in attention_files])

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,

In [41]:
t = np.load("/home/jovyan/hallu_data/hallu_detection_features/helm/filtrations/433_full.npy")
print(t.shape)
print(t[0, 0, :, :2])

(32, 32, 7, 6)
[[ 59.     59.   ]
 [  1.      1.   ]
 [645.    227.   ]
 [ 21.86    7.695]
 [ 43.     31.   ]
 [  1.      1.   ]
 [587.    169.   ]]


In [42]:
len(os.listdir(save_fldr))

204

In [43]:
len(attention_files)

204

# generate barcode features for full output, one colour

In [21]:
import json
from collections import defaultdict

In [19]:
def get_only_barcodes(adj_matricies, ntokens_array, dim, lower_bound):
    """Get barcodes from adj matricies for each layer, head"""
    barcodes = {}
    layers, heads = range(adj_matricies.shape[1]), range(adj_matricies.shape[2])
    for (layer, head) in itertools.product(layers, heads):
        matricies = adj_matricies[:, layer, head, :, :]
        barcodes[(layer, head)] = get_barcodes(matricies, ntokens_array, dim, lower_bound, (layer, head))
    return barcodes

def format_barcodes(barcodes):
    """Reformat barcodes to json-compatible format"""
    return [{d: b[d].tolist() for d in b} for b in barcodes]

def save_barcodes(barcodes, filename):
    """Save barcodes to file"""
    formatted_barcodes = defaultdict(dict)
    for layer, head in barcodes:
        formatted_barcodes[layer][head] = format_barcodes(barcodes[(layer, head)])
    json.dump(formatted_barcodes, open(filename, 'w'))

In [25]:
def process_file_barcode(fname, pth_ats, save_fldr):
    id = fname.split("_")[0]
    try:
        ats = np.load(pth_ats + f"{id}_attentions.npy")
        token_ids = np.load(pth_ats + f"{id}_token_ids.npy")
        hidden_states = np.load(pth_ats + f"{id}_hidden_states.npy")
        params = np.load(pth_ats + f"{id}_params.npy")

        at_cropped = ats[:, :, params[0]:params[3], params[0]:params[3]]
        barcode = get_only_barcodes(at_cropped[None,], [at_cropped.shape[2] for i in range(at_cropped.shape[2])], dim=1, lower_bound=1e-3)
        save_barcodes(barcode, save_fldr + f"{id}_full.json")
    except Exception as e:
        print(f"failed for id {id}, error: {e}")

In [28]:
save_fldr = "/home/jovyan/hallu_data/hallu_detection_features/helm/barcodes/"
for fname in tqdm(attention_files[10:]):
    process_file_barcode(fname, pth_ats, save_fldr)

  0%|          | 0/194 [00:00<?, ?it/s]

# generate filtering features for the first sentence only, one colour

869 - token_id for full stop sign

In [46]:
def process_file_first_sentence(fname, pth_ats, thresholds_array, stats_name, save_fldr):
    id = fname.split("_")[0]
    try:
        ats = np.load(pth_ats + f"{id}_attentions.npy")
        token_ids = np.load(pth_ats + f"{id}_token_ids.npy")
        hidden_states = np.load(pth_ats + f"{id}_hidden_states.npy")
        params = np.load(pth_ats + f"{id}_params.npy")

        prev = params[1]
        stops = np.argwhere(token_ids == 869)
        end = prev
        for s in stops:
            if s > prev + 1:
                end = s[0]
                break
        if end > prev:
            at_cropped = ats[:, :, params[0]:end, params[0]:end]
        else:
            at_cropped = ats[:, :, params[0]:params[3], params[0]:params[3]]
        res = count_top_stats(at_cropped[None,], thresholds_array, [at_cropped.shape[2] for i in range(at_cropped.shape[2])], stats_name, 500, verbose=False)
        res = res.squeeze()
        np.save(save_fldr + f"{id}_first_sent.npy", res)
    except Exception as e:
        print(f"failed for id {id}, error: {e}")

In [47]:
save_fldr = "/home/jovyan/hallu_data/hallu_detection_features/helm/filtrations/"

pool = Pool(10)
pool.starmap(process_file_first_sentence, tqdm([(fname, pth_ats, thresholds_array, stats_name, save_fldr) for fname in attention_files]))

  0%|          | 0/204 [00:00<?, ?it/s]

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,

In [48]:
t = np.load("/home/jovyan/hallu_data/hallu_detection_features/helm/filtrations/433_first_sent.npy")
print(t.shape)
print(t[0, 0, :, :2])

(32, 32, 7, 6)
[[ 179.     179.   ]
 [   7.      45.   ]
 [1149.     383.   ]
 [  12.836    4.28 ]
 [ 112.      74.   ]
 [   7.      45.   ]
 [ 977.     249.   ]]


In [49]:
len(os.listdir(save_fldr))

408

# generate barcode features for the first sentence only, one colour

In [29]:
def process_file_barcode_first_sent(fname, pth_ats, save_fldr):
    id = fname.split("_")[0]
    try:
        ats = np.load(pth_ats + f"{id}_attentions.npy")
        token_ids = np.load(pth_ats + f"{id}_token_ids.npy")
        hidden_states = np.load(pth_ats + f"{id}_hidden_states.npy")
        params = np.load(pth_ats + f"{id}_params.npy")

        prev = params[1]
        stops = np.argwhere(token_ids == 869)
        end = prev
        for s in stops:
            if s > prev + 1:
                end = s[0]
                break
        if end > prev:
            at_cropped = ats[:, :, params[0]:end, params[0]:end]
        else:
            at_cropped = ats[:, :, params[0]:params[3], params[0]:params[3]]
        barcode = get_only_barcodes(at_cropped[None,], [at_cropped.shape[2] for i in range(at_cropped.shape[2])], dim=1, lower_bound=1e-3)
        save_barcodes(barcode, save_fldr + f"{id}_first_sent.json")
    except Exception as e:
        print(f"failed for id {id}, error: {e}")

In [30]:
save_fldr = "/home/jovyan/hallu_data/hallu_detection_features/helm/barcodes/"
for fname in tqdm(attention_files[10:]):
    process_file_barcode_first_sent(fname, pth_ats, save_fldr)

  0%|          | 0/194 [00:00<?, ?it/s]